# Lesson 2.5d — 生产级 RAG 流水线（统一索引 + Rerank）

**目标**：把 `05a` 产出的正文 + 表格 chunk **统一入库**，走「向量召回 → Cross-Encoder Rerank → LLM 一次回答」的生产链路。

与 `05c`（分路径对比）不同，本 Notebook **只建一套索引、只答一次**。

| 组件 | 选型 | 说明 |
|------|------|------|
| 语料 | `05a` JSONL | 正文 `text` + 表格 `table` 统一 chunk |
| Embedding | FastEmbed `all-MiniLM-L6-v2` | ONNX，避开 Windows transformers 长路径问题 |
| 向量库 | FAISS | 先召回 `RETRIEVE_K` 条 |
| Rerank | FastEmbed `ms-marco-MiniLM-L-6-v2` | Cross-Encoder 精排 → `RERANK_K` 条 |
| LLM | DeepSeek API | 需 `DEEPSEEK_API_KEY` |

前置：[05a_part1_pipeline.ipynb](05a_part1_pipeline.ipynb)

---

```mermaid
flowchart LR
    A[05a JSONL<br/>text + table] --> B[统一 Document 列表]
    B --> C[FastEmbed 向量化]
    C --> D[FAISS 索引]
    Q[用户问题] --> D
    D --> E[粗召回 Top-20]
    E --> F[Cross-Encoder Rerank]
    F --> G[精排 Top-5]
    G --> H[拼接 Context]
    H --> I[DeepSeek 生成答案]

    style A fill:#1565C0,color:#fff
    style D fill:#00838F,color:#fff
    style F fill:#E65100,color:#fff
    style I fill:#2E7D32,color:#fff
```

## 0. 安装依赖

In [10]:
## %pip install langchain-openai langchain-community faiss-cpu python-dotenv fastembed -q

## 1. 配置与路径

In [11]:
import json
import os
import re
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.documents import Document

load_dotenv(override=True)

RETRIEVE_K = 20   # 向量粗召回数量
RERANK_K = 5      # Rerank 后送入 LLM 的数量
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
# bge-reranker-base 在部分 Windows 环境 ONNX 下载不完整，改用轻量 ms-marco
RERANK_MODEL = "Xenova/ms-marco-MiniLM-L-6-v2"


def _resolve_here() -> Path:
    for base in (Path.cwd(), Path.cwd() / "02-RAG" / "05_PDF_process"):
        if (base / "doc" / "attention_is_all_you_need.pdf").exists():
            return base.resolve()
    return Path.cwd().resolve()


HERE = _resolve_here()
PDF_PATH = HERE / "doc" / "attention_is_all_you_need.pdf"
CHUNKS_JSONL = HERE / "out" / "chunks" / f"{PDF_PATH.stem}_chunks.jsonl"
TABLE_MD_DIR = HERE / "out" / "markdown"
FASTEMBED_CACHE = HERE / ".cache" / "fastembed"

print(f"工作目录: {HERE}")
print(f"chunks: {CHUNKS_JSONL}")

工作目录: D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\02-RAG\05_PDF_process
chunks: D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\02-RAG\05_PDF_process\out\chunks\attention_is_all_you_need_chunks.jsonl


## 2. 加载统一语料（正文 + 表格）

`05a` 的 JSONL 已包含 `source_type=text|table|ocr`；若 JSONL 里没有表格 chunk，则从 `03` 的 `out/markdown/` 补全。

In [12]:
@dataclass
class Chunk:
    file: str
    page: int
    chunk_id: int
    text: str
    token_count: int
    source_type: str


def load_chunks(jsonl_path: Path) -> list[Chunk]:
    if not jsonl_path.exists():
        raise FileNotFoundError(f"请先运行 05a: {jsonl_path}")
    chunks = []
    for line in jsonl_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            chunks.append(Chunk(**json.loads(line)))
    return chunks


def supplement_table_chunks(chunks: list[Chunk], md_dir: Path) -> list[Chunk]:
    """若 05a 未抽到表格，从 03 的 markdown 目录补充。"""
    if any(c.source_type == "table" for c in chunks) or not md_dir.exists():
        return chunks
    next_id = max((c.chunk_id for c in chunks), default=-1) + 1
    extra = []
    for p in sorted(md_dir.glob("*.md")):
        text = p.read_text(encoding="utf-8").strip()
        if not text:
            continue
        m = re.match(r"plumber_p(\d+)_t(\d+)", p.stem)
        page = int(m.group(1)) if m else 0
        extra.append(Chunk(PDF_PATH.name, page, next_id, text, len(text.split()), "table"))
        next_id += 1
    print(f"从 {md_dir} 补充 {len(extra)} 个表格 chunk")
    return chunks + extra


def chunks_to_documents(chunks: list[Chunk]) -> list[Document]:
    return [
        Document(
            page_content=c.text,
            metadata={
                "file": c.file,
                "page": c.page,
                "chunk_id": c.chunk_id,
                "source_type": c.source_type,
            },
        )
        for c in chunks
    ]


all_chunks = supplement_table_chunks(load_chunks(CHUNKS_JSONL), TABLE_MD_DIR)
documents = chunks_to_documents(all_chunks)

from collections import Counter
cnt = Counter(c.source_type for c in all_chunks)
print(f"共 {len(documents)} 个 chunk: {dict(cnt)}")

共 46 个 chunk: {'text': 36, 'table': 10}


## 3. Embedding + FAISS 索引

In [13]:
from fastembed import TextEmbedding
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings


class FastEmbedEmbeddings(Embeddings):
    def __init__(self, model_name: str = EMBED_MODEL):
        self._model = TextEmbedding(model_name)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [list(v) for v in self._model.embed(texts)]

    def embed_query(self, text: str) -> list[float]:
        return list(next(self._model.embed([text])))


embeddings = FastEmbedEmbeddings()
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVE_K})

print(f"FAISS 索引完成: {len(documents)} docs, 粗召回 k={RETRIEVE_K}")

FAISS 索引完成: 46 docs, 粗召回 k=20


## 4. Cross-Encoder Rerank

In [14]:
from fastembed.rerank.cross_encoder import TextCrossEncoder

_reranker: TextCrossEncoder | None = None


def get_reranker() -> TextCrossEncoder:
    """懒加载 Reranker，缓存到项目目录避免 Temp 路径问题。"""
    global _reranker
    if _reranker is None:
        FASTEMBED_CACHE.mkdir(parents=True, exist_ok=True)
        _reranker = TextCrossEncoder(
            model_name=RERANK_MODEL,
            cache_dir=str(FASTEMBED_CACHE),
        )
    return _reranker


def rerank_documents(query: str, docs: list[Document], top_k: int = RERANK_K) -> list[tuple[float, Document]]:
    if not docs:
        return []
    texts = [d.page_content for d in docs]
    scores = list(get_reranker().rerank(query, texts))
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return ranked[:top_k]


def retrieve_and_rerank(query: str) -> tuple[list[Document], list[tuple[float, Document]]]:
    """返回 (粗召回 docs, rerank 后 [(score, doc), ...])"""
    coarse_docs = retriever.invoke(query)
    reranked = rerank_documents(query, coarse_docs, top_k=RERANK_K)
    return coarse_docs, reranked

## 5. LLM 生成（一次回答）

In [15]:
from langchain_openai import ChatOpenAI

DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")


def format_context(reranked: list[tuple[float, Document]]) -> str:
    parts = []
    for i, (score, doc) in enumerate(reranked):
        meta = doc.metadata
        header = (
            f"[{i}] score={score:.3f} | page={meta.get('page')} "
            f"chunk_id={meta.get('chunk_id')} type={meta.get('source_type')}"
        )
        parts.append(f"{header}\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


def answer_with_rag(query: str, use_rerank: bool = True) -> str:
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("未设置 DEEPSEEK_API_KEY")

    coarse_docs, reranked = retrieve_and_rerank(query)
    if use_rerank:
        selected = [doc for _, doc in reranked]
        context = format_context(reranked)
    else:
        selected = coarse_docs[:RERANK_K]
        context = format_context([(0.0, d) for d in selected])

    llm = ChatOpenAI(
        model=DEEPSEEK_MODEL,
        temperature=0,
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url="https://api.deepseek.com",
    )
    prompt = (
        "根据以下 context 回答问题。必须引用 page 和 chunk_id。\n\n"
        f"Context:\n{context}\n\nQuestion: {query}"
    )
    resp = llm.invoke(prompt)
    return resp.content

C:\Users\86137\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\triton\windows_utils.py:372: UserWarning: Failed to find CUDA.
  warnings.warn("Failed to find CUDA.")


## 6. 运行 & 对比 Rerank 效果

In [16]:
QUESTION = "Transformer base model 的训练步数和 dropout 是多少？"
# 通用问题可改为: "这篇论文的主要内容是什么？"

coarse_docs, reranked = retrieve_and_rerank(QUESTION)

print("=" * 60)
print("粗召回 Top-5（向量相似度，未 Rerank）")
for i, d in enumerate(coarse_docs[:5]):
    m = d.metadata
    print(f"  [{i}] p{m['page']} #{m['chunk_id']} {m['source_type']} | {d.page_content[:80]}...")

print("\n" + "=" * 60)
print("Rerank 后 Top-5（Cross-Encoder 分数）")
for i, (score, d) in enumerate(reranked):
    m = d.metadata
    print(f"  [{i}] score={score:.3f} p{m['page']} #{m['chunk_id']} {m['source_type']} | {d.page_content[:80]}...")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

onnx/model.onnx:   0%|          | 0.00/91.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/824 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

C:\Users\86137\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\02-RAG\05_PDF_process\.cache\fastembed\models--Xenova--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-deve

tokenizer.json: 0.00B [00:00, ?B/s]

2026-07-02 11:20:22.809 | ERROR    | fastembed.common.model_management:download_model:446 - Could not download model from HuggingFace: [WinError 1314] 客户端没有所需的特权。: '..\\..\\blobs\\75305659f7795d4549f0e23688b52fa20a32f925' -> 'D:\\workspace\\doc\\面试狂魔\\人工智能面试题\\AI_coding_interview\\02-RAG\\05_PDF_process\\.cache\\fastembed\\models--Xenova--ms-marco-MiniLM-L-6-v2\\snapshots\\a09144355adeed5f58c8ed011d209bf8ee5a1fec\\tokenizer_config.json' Falling back to other sources.
2026-07-02 11:20:22.810 | ERROR    | fastembed.common.model_management:download_model:469 - Could not download model from either source, sleeping for 3.0 seconds, 2 retries left.


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

粗召回 Top-5（向量相似度，未 Rerank）
  [0] p8 #19 text |  positionalencodingsinboththeencoderanddecoderstacks. Forthebasemodel,weusearate...
  [1] p9 #24 table | | train
N d d h d d P ϵ
model ff k v drop ls steps |
| --- |
| 6 512 2048 8 64 6...
  [2] p9 #21 text | Table3: VariationsontheTransformerarchitecture. Unlistedvaluesareidenticaltothos...
  [3] p2 #4 text | andhavebeenshowntoperformwellonsimple-languagequestionansweringand languagemodel...
  [4] p8 #18 text | Table2: TheTransformerachievesbetterBLEUscoresthanpreviousstate-of-the-artmodels...

Rerank 后 Top-5（Cross-Encoder 分数）
  [0] score=0.226 p8 #19 text |  positionalencodingsinboththeencoderanddecoderstacks. Forthebasemodel,weusearate...
  [1] score=-0.308 p8 #18 text | Table2: TheTransformerachievesbetterBLEUscoresthanpreviousstate-of-the-artmodels...
  [2] score=-1.408 p2 #4 text | andhavebeenshowntoperformwellonsimple-languagequestionansweringand languagemodel...
  [3] score=-1.574 p9 #21 text | Table3: VariationsontheTransformerarch

In [17]:
print("\n" + "=" * 60)
print("【有 Rerank】")
print(answer_with_rag(QUESTION, use_rerank=True))

print("\n" + "=" * 60)
print("【无 Rerank】（直接用向量 Top-5）")
print(answer_with_rag(QUESTION, use_rerank=False))


【有 Rerank】
根据提供的 context，Transformer base model 的训练步数是 **100K**，dropout 率是 **0.1**。

- 训练步数：在 page=9, chunk_id=21 的表格中，列出了 base model 的配置，其中 `train PPL` 列对应的 `steps` 值为 `100K`（见 `page=9 chunk_id=21`）。
- dropout 率：在 page=8, chunk_id=19 中明确提到：“Forthebasemodel,weusearateof P =0.1. drop”（见 `page=8 chunk_id=19`）。

【无 Rerank】（直接用向量 Top-5）
根据提供的 context，Transformer base model 的训练步数是 **100K**，dropout 率是 **0.1**。

- 训练步数：在 [1] 中，表格显示 base 模型的 `train steps` 为 `100K`（page=9, chunk_id=24）。
- dropout 率：在 [0] 中，明确提到“For the base model, we use a rate of P = 0.1”（page=8, chunk_id=19）。


## 7. 小结

| 阶段 | 教学 Notebook | 生产实践（本 Notebook） |
|------|--------------|------------------------|
| 解析入库 | 05a 统一 JSONL | 同左，正文+表格一个库 |
| 检索 | 05c 分路径各建索引 | **一套 FAISS** |
| 精排 | 无 | **Cross-Encoder Rerank** |
| 生成 | 分路径各答一次 | **一次 LLM 调用** |

**调参建议**：
- 召回噪声多 → 增大 `RETRIEVE_K` 或换更强 Reranker
- Context 太长 → 减小 `RERANK_K`
- 表格问题偏多 → 确保 `source_type=table` chunk 已入库（05a 或 03 补充）

**与 05c 的关系**：05c 用于**选型对比**；本 Notebook 是选型后的**上线形态**。